# Karpathy model ailesi — tek FPGA motorunda canlı demo

PYNQ-Z2 üzerinde **tek bir bitstream**: modeller çalışma anında yüklenir, motor bit-exact tamsayı çıkarım yapar.
Önce `docs/SENIN_YAPACAKLARIN.md` K3 adımıyla depoyu ve `zoo.bit`/`zoo.hwh` dosyalarını karta koy.

In [ ]:
import os, sys, time
ROOT = '/home/xilinx/mingpt-fpga'          # depo konumu
sys.path.insert(0, os.path.join(ROOT, 'board', 'pynq_z2'))
from zoo import Zoo
from run_board import decode
z = Zoo(os.path.join(ROOT, 'build', 'zoo', 'pynq_overlay', 'zoo.bit'))
print(f'motor: V={z.V}, T={z.T}, C={z.C}, P={z.P}, katman={z.NL}')

## 1. İsim üretimi (names.txt, ~50K parametre)
Model seç: `bigram, mlp, rnn, gru, bow, gpt2, microgpt, llama, nanochat`.

In [ ]:
import random, json
ITOS = '.abcdefghijklmnopqrstuvwxyz'
def isim(model, n=8, sicaklik=True):
    d = os.path.join(ROOT, 'mem', 'zoo', 'universal', model)
    z.load(d)
    s_logit = json.load(open(os.path.join(d, 'dims.json')))['S_LOGIT']   # one INT16 logit step in nats
    out = []
    for _ in range(n):
        tok, s = 0, ''
        for t in range(16):
            am, cyc = z.step(tok, t)
            if sicaklik:                       # logitlerden örnekleme (INT16 logit -> olasılık)
                import math
                lg = z.logits(27); m = max(lg); w = [math.exp((v - m) * s_logit) for v in lg]
                tok = random.choices(range(27), weights=w)[0]
            else:
                tok = am
            if tok == 0: break
            s += ITOS[tok]
        out.append(s)
    return out, cyc
names, cyc = isim('gru')
print(names, f'{cyc} çevrim/token')

## 2. TinyStories hikâyesi (stories260K, Llama 2, 260K parametre)

In [ ]:
z.load(os.path.join(ROOT, 'mem', 'zoo', 'universal', 'stories260k'))
t0 = time.perf_counter()
toks, cyc = z.generate(200, start=1)
dt = time.perf_counter() - t0
print(decode(toks))
print(f'\n{len(toks)-1} token, ortalama {sum(cyc)/len(cyc):.0f} çevrim/token, {len(cyc)/dt:.1f} token/s (Python dahil)')

## 3. Token başına süre konumla nasıl değişir?
Dikkat (attention) her konumda bütün önceki konumları okur: süre konumla doğrusal artar.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(cyc); plt.xlabel('konum t'); plt.ylabel('çevrim / token'); plt.title('stories260K, FPGA motoru'); plt.show()